# SQL Server Troubleshooting Notebook

Runs against one or more servers using dbatools. Edit the Parameters cell before executing.

> **Sections**
> - Parameters
> - Blocking & Lock Analysis
> - Wait Statistics
> - Long-Running Queries
> - Open Transactions

In [ ]:
# Parameters
$SqlInstances = [System.Collections.ArrayList]@()

# Local / Dev
#$SqlInstances.Add([pscustomobject]@{Instance='localhost'}) | Out-Null

# Production
#$SqlInstances.Add([pscustomobject]@{Instance='SERVER01'}) | Out-Null
#$SqlInstances.Add([pscustomobject]@{Instance='SERVER02'}) | Out-Null
#$SqlInstances.Add([pscustomobject]@{Instance='SERVER03'}) | Out-Null

$SqlInstances.Add([pscustomobject]@{Instance='GP-ENT-NEW\ENT'}) | Out-Null
#$SqlInstances.Add([pscustomobject]@{Instance='SQL-RPL-NEW\RPL'}) | Out-Null
#$SqlInstances.Add([pscustomobject]@{Instance='WMS-SQL\ODNWMS'}) | Out-Null
#$SqlInstances.Add([pscustomobject]@{Instance='SQL-PMA\ODNPMA'}) | Out-Null

# Test
#$SqlInstances.Add([pscustomobject]@{Instance='SERVER01'}) | Out-Null

#$SqlInstances.Add([pscustomobject]@{Instance='SQL-ENT-TEST\ENT'}) | Out-Null
$SqlInstances.Add([pscustomobject]@{Instance='SQL-RPL-TEST\RPL'}) | Out-Null
#$SqlInstances.Add([pscustomobject]@{Instance='TEST-WMS-SQL'}) | Out-Null

# Troubleshooting Parameters
$BlockingThresholdSeconds = 5
$TopN                     = 25
$Credential               = $null   # Set to Get-Credential for SQL auth

$SqlInstances | Format-Table -AutoSize

## Prerequisites

Verifies that dbatools is installed locally and that the required stored procedures are present
on each instance before running any sections below. Missing procedures can be installed using
the commands shown in the warnings.

| Procedure | Purpose |
|---|---|
| `sp_WhoIsActive` | Real-time session and blocking analysis |
| `sp_Blitz` | Instance health check |
| `sp_BlitzFirst` | Point-in-time wait stats and performance snapshot |
| `sp_BlitzWho` | Current activity snapshot with blocking detail |

In [ ]:
# Verify dbatools is installed
$dbatools = Get-Module -Name dbatools -ListAvailable
if ($dbatools) {
    Write-Host "dbatools found — version $($dbatools[0].Version)" -ForegroundColor Green
} else {
    Write-Warning "dbatools not found — run: Install-Module dbatools -Scope CurrentUser"
    return
}

# Verify sp_WhoIsActive and First Responder Kit on each instance
$procsToCheck = @(
    @{ Name = 'sp_WhoIsActive'; InstallCmd = 'Install-DbaWhoIsActive -SqlInstance {0} -Database master' },
    @{ Name = 'sp_Blitz';       InstallCmd = 'Install-DbaFirstResponderKit -SqlInstance {0} -Database master' },
    @{ Name = 'sp_BlitzFirst';  InstallCmd = 'Install-DbaFirstResponderKit -SqlInstance {0} -Database master' },
    @{ Name = 'sp_BlitzWho';    InstallCmd = 'Install-DbaFirstResponderKit -SqlInstance {0} -Database master' }
)

foreach ($instance in $SqlInstances) {
    Write-Host "`n=== $($instance.Instance) ===" -ForegroundColor Cyan

    foreach ($proc in $procsToCheck) {
        $exists = Get-DbaDbStoredProcedure -SqlInstance $instance.Instance `
                      -Database master -Name $proc.Name

        if ($exists) {
            Write-Host "  $($proc.Name) found" -ForegroundColor Green
        } else {
            Write-Warning "  $($proc.Name) NOT found — run: $($proc.InstallCmd -f $instance.Instance)"
        }
    }
}

## Navigation

- [Prerequisites](#prerequisites)
- [Blocking & Lock Analysis](#blocking--lock-analysis)
- [Wait Statistics](#wait-statistics)
- [Long-Running Queries](#long-running-queries)
- [Open Transactions](#open-transactions)
- [Deadlock Analysis](#deadlock-analysis)
- [Top Resource-Consuming Queries](#top-resource-consuming-queries)
- [TempDB Contention](#tempdb-contention)
- [Database File Space](#database-file-space)
- [Agent Job Status](#agent-job-status)

### Quick Blocking Check

Runs against all instances in the parameter block. Returns any sessions that are currently
blocked or acting as a head blocker. Columns are kept minimal for fast triage — if this
returns results, run the Full Chain cell below for deeper detail.

In [ ]:
foreach ($instance in $SqlInstances) {
    Write-Host "`n=== $($instance.Instance) ===" -ForegroundColor Cyan

    $blocked = Invoke-DbaWhoIsActive -SqlInstance $instance.Instance -FindBlockLeaders |
        Where-Object { ($_.blocking_session_id -as [int]) -gt 0 -or ($_.blocked_session_count -as [int]) -gt 0 }

    if ($blocked) {
        $blocked |
            Select-Object session_id, blocking_session_id, blocked_session_count,
                          wait_type, wait_time, database_name, login_name, [sql_text] |
            Format-Table -AutoSize
    } else {
        Write-Host "  No blocking detected" -ForegroundColor Green
    }
}

### Full Blocking Chain Detail

Run this after the Quick Check confirms blocking is present. Adds lock detail, host name,
and query text to help identify the head blocker and what it's holding. The `[locks]` column
returns XML — paste it into SSMS for easier reading.

> **Note:** `GetTaskInfo = 2` includes parallel query worker threads, which can add noise.
> If output is cluttered, remove `-GetTaskInfo 2` to show only the lead session.

In [ ]:
foreach ($instance in $SqlInstances) {
    Write-Host "`n=== $($instance.Instance) ===" -ForegroundColor Cyan

    $blocked = Invoke-DbaWhoIsActive -SqlInstance $instance.Instance `
        -GetLocks -FindBlockLeaders -GetTaskInfo 2 -As PSObject |
        Where-Object { ($_.blocking_session_id -as [int]) -gt 0 -or ($_.blocked_session_count -as [int]) -gt 0 }

    if ($blocked) {
        $blocked |
            Select-Object session_id, blocking_session_id, blocked_session_count,
                          wait_info, database_name, login_name,
                          host_name, sql_text, locks |
            Out-String -Width 512 |
            Write-Host
    } else {
        Write-Host "  No blocking detected" -ForegroundColor Green
    }
}

## Wait Statistics

Wait statistics show what SQL Server is spending time waiting on. Use this as a starting point
for any performance investigation — the top waits tell you where to focus next.

`Get-DbaWaitStatistic` filters out benign idle waits automatically and returns cumulative stats
since the last service restart or manual reset. Use `sp_BlitzFirst` for a point-in-time sample
over a short interval rather than cumulative totals.

> **Note:** Cumulative waits can be misleading on long-running instances. If the server has been
> up for months, a one-time event may still show near the top. Use `sp_BlitzFirst` to confirm
> whether a wait type is actively occurring right now.

In [ ]:
foreach ($instance in $SqlInstances) {
    Write-Host "`n=== $($instance.Instance) ===" -ForegroundColor Cyan

    Get-DbaWaitStatistic -SqlInstance $instance.Instance -Threshold 95 |
        Select-Object WaitType, WaitCount, WaitSeconds, ResourceSeconds,
                      SignalSeconds, Percentage |
        Format-Table -AutoSize |
        Out-String -Width 512 |
        Write-Host
}

### Point-in-Time Wait Snapshot (sp_BlitzFirst)

> **⚠️ Runtime warning:** `sp_BlitzFirst` can take 2+ minutes to return depending on instance
> load and the `-Seconds` sample interval. Do not run during an active incident without
> accounting for the wait. Consider running it after stabilizing the environment.
>
> Uncomment and run manually when you have time to wait for results.

In [ ]:
foreach ($instance in $SqlInstances) {
    Write-Host "`n=== $($instance.Instance) ===" -ForegroundColor Cyan

    Invoke-DbaQuery -SqlInstance $instance.Instance -Database master `
        -Query "EXEC sp_BlitzFirst @Seconds = 5, @ExpertMode = 0;" |
        Select-Object Priority, FindingsGroup, Finding, Details |
        Out-String -Width 512 |
        Write-Host
}

## Long-Running Queries

Identifies queries that have been executing longer than the threshold set in the Parameters cell.
Results are sorted by elapsed time descending so the worst offenders are at the top.

Use this alongside the blocking section — a long-running query is often either the cause of
blocking or is itself being blocked. Check the `blocking_session_id` column to distinguish
between the two.

### Currently Running Queries Exceeding Threshold

In [ ]:
foreach ($instance in $SqlInstances) {
    Write-Host "`n=== $($instance.Instance) ===" -ForegroundColor Cyan

    $longRunning = Invoke-DbaWhoIsActive -SqlInstance $instance.Instance `
        -FindBlockLeaders -GetOuterCommand -As PSObject |
        Where-Object { ($_.elapsed_time -as [int]) -gt ($BlockingThresholdSeconds * 1000) }

    if ($longRunning) {
        $longRunning |
            Select-Object session_id, elapsed_time, blocking_session_id,
                          database_name, login_name, host_name,
                          sql_text, sql_command |
            Format-Table -AutoSize |
            Out-String -Width 512 |
            Write-Host
    } else {
        Write-Host "  No queries exceeding $BlockingThresholdSeconds seconds" -ForegroundColor Green
    }
}

## Open Transactions

Identifies sessions with open transactions that have not been committed or rolled back.
Long-running open transactions hold locks, consume transaction log space, and are a common
cause of blocking and log growth.

Sleeping sessions with open transactions are particularly dangerous — the application may have
abandoned the transaction entirely, leaving locks in place indefinitely until the session is
killed or times out.

### Sessions with Open Transactions

In [ ]:
foreach ($instance in $SqlInstances) {
    Write-Host "`n=== $($instance.Instance) ===" -ForegroundColor Cyan

    $openTrans = Invoke-DbaWhoIsActive -SqlInstance $instance.Instance `
        -GetTransactionInfo -ShowSleepingSpids 2 -As PSObject |
        Where-Object { ($_.tran_start_time -ne $null) -and ($_.tran_start_time -ne '') }

    if ($openTrans) {
        $openTrans |
            Select-Object session_id, status, blocking_session_id,
                          tran_start_time, tran_log_used_percent,
                          database_name, login_name, host_name, sql_text |
            Format-Table -AutoSize -Wrap |
            Out-String -Width 1024 |
            Write-Host
    } else {
        Write-Host "  No open transactions detected" -ForegroundColor Green
    }
}

## Deadlock Analysis

Queries `DBAOps.trace.DeadlockHistory` which is populated every 5 minutes from the
`system_health` ring buffer via the `DBAOps - Capture Deadlocks` Agent job.

Use this section to identify deadlock patterns, frequency, and involved sessions after
a deadlock complaint. For raw XML graph analysis, paste the `DeadlockGraph` column
output into SSMS for the visual deadlock graph viewer.

### Recent Deadlocks

In [ ]:
foreach ($instance in $SqlInstances) {
    Write-Host "`n=== $($instance.Instance) ===" -ForegroundColor Cyan

    $deadlocks = Invoke-DbaQuery -SqlInstance $instance.Instance -Database DBAOps -Query "
        SELECT TOP (@TopN)
            [DeadlockId],
            [EventTime],
            [DatabaseName],
            [VictimProcess],
            [CapturedAt]
        FROM [trace].[DeadlockHistory]
        ORDER BY [EventTime] DESC;
    " -SqlParameter @{ TopN = $TopN }

    if ($deadlocks) {
        $deadlocks |
            Format-Table -AutoSize |
            Out-String -Width 512 |
            Write-Host
    } else {
        Write-Host "  No deadlocks recorded" -ForegroundColor Green
    }
}

### Deadlock Frequency by Database

Useful for identifying which databases are experiencing the most deadlocks over time.

In [ ]:
foreach ($instance in $SqlInstances) {
    Write-Host "`n=== $($instance.Instance) ===" -ForegroundColor Cyan

    $frequency = Invoke-DbaQuery -SqlInstance $instance.Instance -Database DBAOps -Query "
        SELECT
            [DatabaseName],
            COUNT(*)                AS DeadlockCount,
            MIN([EventTime])        AS FirstSeen,
            MAX([EventTime])        AS LastSeen
        FROM [trace].[DeadlockHistory]
        GROUP BY [DatabaseName]
        ORDER BY DeadlockCount DESC;
    "

    if ($frequency) {
        $frequency |
            Format-Table -AutoSize |
            Out-String -Width 512 |
            Write-Host
    } else {
        Write-Host "  No deadlocks recorded" -ForegroundColor Green
    }
}

### Deadlock Graph XML

Returns the raw deadlock graph XML for a specific `DeadlockId`. Paste the XML into
SSMS to view the visual deadlock graph. Set `$DeadlockId` to the ID from the Recent
Deadlocks cell above.

In [ ]:
$DeadlockId     = 1         # <-- set to target DeadlockId
$TargetServer   = $SqlInstances[0].Instance   # <-- confirm which server

Invoke-DbaQuery -SqlInstance $TargetServer -Database DBAOps -Query "
    SELECT
        [DeadlockId],
        [EventTime],
        [DatabaseName],
        [VictimProcess],
        [DeadlockGraph]
    FROM [trace].[DeadlockHistory]
    WHERE [DeadlockId] = @DeadlockId;
" -SqlParameter @{ DeadlockId = $DeadlockId } |
    Out-String -Width 2048 |
    Write-Host

## Top Resource-Consuming Queries

Pulls from the plan cache (`sys.dm_exec_query_stats`). Results reflect cumulative stats since
the last plan cache flush or service restart — a query at the top may have run millions of times
over weeks rather than being a current problem. Cross-reference with the blocking and wait stats
sections to confirm whether a query is actively causing issues right now.

> **Note:** Plan cache entries can be evicted at any time under memory pressure. If a known
> problem query isn't appearing, it may have been flushed.

### Top Queries by CPU

In [ ]:
$ExcludeInternalQueries   = $false     # Set to $true to hide XE/internal monitoring queries

foreach ($instance in $SqlInstances) {
    Write-Host "`n=== $($instance.Instance) ===" -ForegroundColor Cyan

$excludeFilter = if ($ExcludeInternalQueries) {
    "AND [qt].[text] NOT LIKE '%#XEStaging%'
     AND [qt].[text] NOT LIKE '%dm_xe_session%'
     AND [qt].[text] NOT LIKE '%ring_buffer%'
     AND [qt].[text] NOT LIKE '%target_data%'
     AND [qt].[text] NOT LIKE '%dm_exec_query_stats%'
     AND [qt].[text] NOT LIKE '%sp_WhoIsActive%'
     AND [qt].[text] NOT LIKE '%sp_Blitz%'
     AND [qt].[text] NOT LIKE '%sysschedules%'
     AND [qt].[text] NOT LIKE '%sysjobhistory%'
     AND [qt].[text] NOT LIKE '%tmp_replication_status%'
     AND [qt].[text] NOT LIKE '%#tmp%'"
} else { "" }

    Invoke-DbaQuery -SqlInstance $instance.Instance -Database master -Query "
        SELECT TOP (@TopN)
            DB_NAME([qt].[dbid])                                AS DatabaseName,
            [qs].[execution_count]                              AS ExecutionCount,
            [qs].[total_worker_time] / 1000000.0                AS TotalCPUSec,
            [qs].[total_worker_time] / [qs].[execution_count] 
                / 1000000.0                                     AS AvgCPUSec,
            [qs].[total_elapsed_time] / [qs].[execution_count]
                / 1000000.0                                     AS AvgElapsedSec,
            SUBSTRING([qt].[text],
                ([qs].[statement_start_offset] / 2) + 1,
                ((CASE [qs].[statement_end_offset]
                    WHEN -1 THEN DATALENGTH([qt].[text])
                    ELSE [qs].[statement_end_offset]
                  END - [qs].[statement_start_offset]) / 2) + 1
            )                                                   AS QueryText
        FROM [sys].[dm_exec_query_stats]            AS qs
        CROSS APPLY [sys].[dm_exec_sql_text]([qs].[sql_handle]) AS qt
        WHERE 1 = 1
        $excludeFilter
        ORDER BY [qs].[total_worker_time] DESC;
    " -SqlParameter @{ TopN = $TopN } |
        Format-Table -AutoSize |
        Out-String -Width 512 |
        Write-Host
}

### Top Queries by Logical Reads

In [ ]:
$ExcludeInternalQueries   = $false     # Set to $true to hide XE/internal monitoring queries

foreach ($instance in $SqlInstances) {
    Write-Host "`n=== $($instance.Instance) ===" -ForegroundColor Cyan

$excludeFilter = if ($ExcludeInternalQueries) {
    "AND [qt].[text] NOT LIKE '%#XEStaging%'
     AND [qt].[text] NOT LIKE '%dm_xe_session%'
     AND [qt].[text] NOT LIKE '%ring_buffer%'
     AND [qt].[text] NOT LIKE '%target_data%'
     AND [qt].[text] NOT LIKE '%dm_exec_query_stats%'
     AND [qt].[text] NOT LIKE '%sp_WhoIsActive%'
     AND [qt].[text] NOT LIKE '%sp_Blitz%'
     AND [qt].[text] NOT LIKE '%sysschedules%'
     AND [qt].[text] NOT LIKE '%sysjobhistory%'
     AND [qt].[text] NOT LIKE '%tmp_replication_status%'
     AND [qt].[text] NOT LIKE '%#tmp%'"
} else { "" }

    Invoke-DbaQuery -SqlInstance $instance.Instance -Database master -Query "
        SELECT TOP (@TopN)
            DB_NAME([qt].[dbid])                                AS DatabaseName,
            [qs].[execution_count]                              AS ExecutionCount,
            [qs].[total_logical_reads]                          AS TotalLogicalReads,
            [qs].[total_logical_reads] / [qs].[execution_count] AS AvgLogicalReads,
            [qs].[total_elapsed_time] / [qs].[execution_count]
                / 1000000.0                                     AS AvgElapsedSec,
            SUBSTRING([qt].[text],
                ([qs].[statement_start_offset] / 2) + 1,
                ((CASE [qs].[statement_end_offset]
                    WHEN -1 THEN DATALENGTH([qt].[text])
                    ELSE [qs].[statement_end_offset]
                  END - [qs].[statement_start_offset]) / 2) + 1
            )                                                   AS QueryText
        FROM [sys].[dm_exec_query_stats]            AS qs
        CROSS APPLY [sys].[dm_exec_sql_text]([qs].[sql_handle]) AS qt
        WHERE 1 = 1
        $excludeFilter        
        ORDER BY [qs].[total_logical_reads] DESC;
    " -SqlParameter @{ TopN = $TopN } |
        Format-Table -AutoSize |
        Out-String -Width 512 |
        Write-Host
}

### Top Queries by Writes

In [ ]:
$ExcludeInternalQueries   = $true     # Set to $true to hide XE/internal monitoring queries

foreach ($instance in $SqlInstances) {
    Write-Host "`n=== $($instance.Instance) ===" -ForegroundColor Cyan

$excludeFilter = if ($ExcludeInternalQueries) {
    "AND [qt].[text] NOT LIKE '%#XEStaging%'
     AND [qt].[text] NOT LIKE '%dm_xe_session%'
     AND [qt].[text] NOT LIKE '%ring_buffer%'
     AND [qt].[text] NOT LIKE '%target_data%'
     AND [qt].[text] NOT LIKE '%dm_exec_query_stats%'
     AND [qt].[text] NOT LIKE '%sp_WhoIsActive%'
     AND [qt].[text] NOT LIKE '%sp_Blitz%'
     AND [qt].[text] NOT LIKE '%sysschedules%'
     AND [qt].[text] NOT LIKE '%sysjobhistory%'
     AND [qt].[text] NOT LIKE '%tmp_replication_status%'
     AND [qt].[text] NOT LIKE '%#tmp%'"
} else { "" }

    Invoke-DbaQuery -SqlInstance $instance.Instance -Database master -Query "
        SELECT TOP (@TopN)
            DB_NAME([qt].[dbid])                                AS DatabaseName,
            [qs].[execution_count]                              AS ExecutionCount,
            [qs].[total_logical_writes]                         AS TotalLogicalWrites,
            [qs].[total_logical_writes] / [qs].[execution_count] AS AvgLogicalWrites,
            [qs].[total_elapsed_time] / [qs].[execution_count]
                / 1000000.0                                     AS AvgElapsedSec,
            SUBSTRING([qt].[text],
                ([qs].[statement_start_offset] / 2) + 1,
                ((CASE [qs].[statement_end_offset]
                    WHEN -1 THEN DATALENGTH([qt].[text])
                    ELSE [qs].[statement_end_offset]
                  END - [qs].[statement_start_offset]) / 2) + 1
            )                                                   AS QueryText
        FROM [sys].[dm_exec_query_stats]            AS qs
        CROSS APPLY [sys].[dm_exec_sql_text]([qs].[sql_handle]) AS qt
        WHERE 1 = 1
        $excludeFilter        
        ORDER BY [qs].[total_logical_writes] DESC;
    " -SqlParameter @{ TopN = $TopN } |
        Format-Table -AutoSize |
        Out-String -Width 512 |
        Write-Host
}

## TempDB Contention

TempDB contention typically manifests as `PAGELATCH_*` waits on PFS, GAM, or SGAM pages.
The standard fix is ensuring TempDB has one data file per logical CPU core (up to 8). 

Sections below cover:
- Active PAGELATCH waits on TempDB allocation pages
- TempDB file configuration
- Active session TempDB space usage
- Version store usage (relevant if any databases use RCSI or snapshot isolation)

### Active TempDB PAGELATCH Waits

PFS pages occur at file offsets 0:1, 0:8088, 0:16176, etc.
GAM pages occur at 0:2 and every ~64,000 pages thereafter.
SGAM pages occur at 0:3 and every ~64,000 pages thereafter.
Contention on these pages indicates TempDB needs more data files.

In [ ]:
foreach ($instance in $SqlInstances) {
    Write-Host "`n=== $($instance.Instance) ===" -ForegroundColor Cyan

    $waits = Invoke-DbaQuery -SqlInstance $instance.Instance -Database master -Query "
        SELECT
            [t].[session_id],
            [t].[exec_context_id],
            [t].[wait_type],
            [t].[wait_duration_ms],
            [t].[resource_description],
            [s].[login_name],
            [s].[host_name],
            [s].[program_name]
        FROM [sys].[dm_os_waiting_tasks]    AS t
        JOIN [sys].[dm_exec_sessions]       AS s
            ON t.[session_id] = s.[session_id]
        WHERE [t].[wait_type] LIKE 'PAGELATCH_%'
          AND [t].[resource_description] LIKE '2:%'
        ORDER BY [t].[wait_duration_ms] DESC;
    "

    if ($waits) {
        $waits |
            Format-Table -AutoSize |
            Out-String -Width 512 |
            Write-Host
    } else {
        Write-Host "  No TempDB PAGELATCH waits detected" -ForegroundColor Green
    }
}

### TempDB File Configuration

Best practice: one data file per logical CPU core, up to 8. All data files should be
equal size with the same autogrowth settings. A single data file or unequal file sizes
will cause allocation page contention under load.

In [ ]:
foreach ($instance in $SqlInstances) {
    Write-Host "`n=== $($instance.Instance) ===" -ForegroundColor Cyan

    Write-Host "`n-- Best Practice Check --" -ForegroundColor Yellow
    Test-DbaTempDbConfig -SqlInstance $instance.Instance |
        Select-Object Rule, CurrentSetting, Recommended, IsBestPractice, Notes |
        Format-Table -AutoSize |
        Out-String -Width 512 |
        Write-Host

    Write-Host "-- File Detail --" -ForegroundColor Yellow
    Invoke-DbaQuery -SqlInstance $instance.Instance -Database tempdb -Query "
        SELECT
            [f].[name]                                                          AS FileName,
            [f].[type_desc]                                                     AS FileType,
            [f].[size] * 8 / 1024                                               AS SizeMB,
            ([f].[size] - FILEPROPERTY([f].[name], 'SpaceUsed')) * 8 / 1024    AS FreeMB,
            CAST(
                (([f].[size] - FILEPROPERTY([f].[name], 'SpaceUsed')) * 100.0)
                / [f].[size] AS decimal(5,1))                                   AS FreePct,
            CASE [f].[is_percent_growth]
                WHEN 1 THEN CAST([f].[growth] AS varchar) + '%'
                ELSE CAST([f].[growth] * 8 / 1024 AS varchar) + ' MB'
            END                                                                 AS AutoGrowth,
            [f].[physical_name]                                                 AS PhysicalPath
        FROM [sys].[database_files] AS f
        ORDER BY [f].[type_desc], [f].[file_id];
    " |
        Format-Table -AutoSize |
        Out-String -Width 512 |
        Write-Host
}

### Active Session TempDB Space Usage

Shows sessions currently allocating space in TempDB. High `UserObjectAllocated` indicates
temp table usage. High `InternalObjectAllocated` indicates sort or hash spills.

In [ ]:
foreach ($instance in $SqlInstances) {
    Write-Host "`n=== $($instance.Instance) ===" -ForegroundColor Cyan

    $usage = Get-DbaTempdbUsage -SqlInstance $instance.Instance |
        Sort-Object { $_.TotalUserAllocatedKB + $_.TotalInternalAllocatedKB } -Descending |
        Select-Object -First $TopN

    if ($usage) {
        $usage |
            Select-Object Spid, LoginName, HostName, Database, Status,
                          TotalUserAllocatedKB, UserDeallocatedKB,
                          TotalInternalAllocatedKB, InternalDeallocatedKB,
                          StatementCommand, QueryText |
            Format-Table -AutoSize |
            Out-String -Width 512 |
            Write-Host
    } else {
        Write-Host "  No active TempDB space usage detected" -ForegroundColor Green
    }
}

### Version Store Usage

The version store grows when snapshot isolation or RCSI is enabled on any database, or when
long-running transactions delay cleanup. Excessive growth can fill TempDB and cause failures
across all databases on the instance.

Check `sys.databases` for RCSI/snapshot isolation enablement if version store usage is high —
the culprit is usually a long-running open transaction holding back cleanup.

In [ ]:
foreach ($instance in $SqlInstances) {
    Write-Host "`n=== $($instance.Instance) ===" -ForegroundColor Cyan

    Write-Host "`n-- Version Store Size --" -ForegroundColor Yellow
    Invoke-DbaQuery -SqlInstance $instance.Instance -Database tempdb -Query "
        SELECT
            [version_store_reserved_page_count]     * 8 / 1024  AS VersionStoreMB,
            [user_object_reserved_page_count]       * 8 / 1024  AS UserObjectMB,
            [internal_object_reserved_page_count]   * 8 / 1024  AS InternalObjectMB
        FROM [sys].[dm_db_file_space_usage]
        WHERE [database_id] = 2;
    " |
        Format-Table -AutoSize |
        Out-String -Width 512 |
        Write-Host

    Write-Host "-- Version Store by Database --" -ForegroundColor Yellow
    $versionByDb = Invoke-DbaQuery -SqlInstance $instance.Instance -Database master -Query "
        SELECT
            DB_NAME([database_id])              AS DatabaseName,
            [reserved_page_count] * 8 / 1024   AS ReservedMB,
            [reserved_space_kb] / 1024          AS ReservedSpaceMB
        FROM [sys].[dm_tran_version_store_space_usage]
        ORDER BY [reserved_page_count] DESC;
    "

    if ($versionByDb) {
        $versionByDb |
            Format-Table -AutoSize |
            Out-String -Width 512 |
            Write-Host
    } else {
        Write-Host "  No version store usage detected" -ForegroundColor Green
    }

    Write-Host "-- Databases with Snapshot Isolation Enabled --" -ForegroundColor Yellow
    $snapDbs = Invoke-DbaQuery -SqlInstance $instance.Instance -Database master -Query "
        SELECT
            [name]                              AS DatabaseName,
            [snapshot_isolation_state_desc]     AS SnapshotState,
            [is_read_committed_snapshot_on]     AS RCSIEnabled
        FROM [sys].[databases]
        WHERE [snapshot_isolation_state] != 0
           OR [is_read_committed_snapshot_on] = 1
        ORDER BY [name];
    "

    if ($snapDbs) {
        $snapDbs |
            Format-Table -AutoSize |
            Out-String -Width 512 |
            Write-Host
    } else {
        Write-Host "  No databases with snapshot isolation enabled" -ForegroundColor Green
    }
}

## Database File Space

Checks disk-level free space, database-level size and usage, and individual file detail.
Warnings are flagged when used space exceeds `$DiskSpaceThresholdPct` percent.

### Disk Free Space

In [ ]:
foreach ($instance in $SqlInstances) {
    Write-Host "`n=== $($instance.Instance) ===" -ForegroundColor Cyan

    $disks = Get-DbaDiskSpace -ComputerName $instance.Instance |
        Select-Object Name, Label,
                      @{N='CapacityGB'; E={[math]::Round([long]$_.Capacity/1GB, 1)}},
                      @{N='FreeGB';     E={[math]::Round([long]$_.Free/1GB, 1)}},
                      @{N='UsedPct';    E={[math]::Round((1 - [long]$_.Free/[long]$_.Capacity) * 100, 1)}}

    foreach ($disk in $disks) {
        if ($disk.UsedPct -ge $DiskSpaceThresholdPct) {
            Write-Host "  ⚠ $($disk.Name) — $($disk.FreeGB)GB free ($($disk.UsedPct)% used)" -ForegroundColor Red
        }
    }

    $disks |
        Format-Table -AutoSize |
        Out-String -Width 512 |
        Write-Host
}

### Database Size Summary

Total size, used space, free space, and recovery model per database.

In [ ]:
foreach ($instance in $SqlInstances) {
    Write-Host "`n=== $($instance.Instance) ===" -ForegroundColor Cyan

    Get-DbaDbFile -SqlInstance $instance.Instance |
        Group-Object Database |
        ForEach-Object {
            $files      = $_.Group
            $totalSize  = ($files | Measure-Object { [long]$_.Size } -Sum).Sum
            $usedSpace  = ($files | Measure-Object { [long]$_.UsedSpace } -Sum).Sum
            $freeSpace  = $totalSize - $usedSpace
            $usedPct    = if ($totalSize -gt 0) { [math]::Round($usedSpace * 100.0 / $totalSize, 1) } else { 0 }

            [PSCustomObject]@{
                DatabaseName  = $_.Name
                TotalSizeMB   = [math]::Round($totalSize / 1MB, 0)
                UsedMB        = [math]::Round($usedSpace / 1MB, 0)
                FreeMB        = [math]::Round($freeSpace / 1MB, 0)
                UsedPct       = $usedPct
            }
        } | Sort-Object TotalSizeMB -Descending |
        Format-Table -AutoSize |
        Out-String -Width 512 |
        Write-Host
}

### Database File Detail

Individual file sizes, free space, and autogrowth settings per database.
Files with percentage-based autogrowth are flagged — best practice is fixed MB growth.

In [ ]:
foreach ($instance in $SqlInstances) {
    Write-Host "`n=== $($instance.Instance) ===" -ForegroundColor Cyan

    Get-DbaDbFile -SqlInstance $instance.Instance |
        Select-Object Database, LogicalName, TypeDescription,
            @{N='SizeMB';       E={[math]::Round([long]$_.Size / 1MB, 0)}},
            @{N='UsedMB';       E={[math]::Round([long]$_.UsedSpace / 1MB, 0)}},
            @{N='FreeMB';       E={[math]::Round([long]$_.AvailableSpace / 1MB, 0)}},
            @{N='UsedPct';      E={
                $s = [long]$_.Size
                if ($s -gt 0) { [math]::Round([long]$_.UsedSpace * 100.0 / $s, 1) } else { 0 }
            }},
            @{N='AutoGrowth';   E={
                if ($_.GrowthType -eq 'Percent') { "$($_.Growth)% ⚠" }
                else { "$([math]::Round([long]$_.NextGrowthEventSize / 1MB, 0)) MB" }
            }},
            PhysicalName |
        Sort-Object Database, TypeDescription |
        Format-Table -AutoSize |
        Out-String -Width 512 |
        Write-Host
}

## Agent Job Status

Checks for failed jobs in the last `$JobHistoryHours` hours and jobs currently running
longer than `$JobRunningMultiplier`x their average duration. Set `$ExcludeJobs` in the
parameter block to suppress known long-running maintenance jobs.

### Failed Jobs

In [ ]:
# Failed Jobs
foreach ($instance in $SqlInstances) {
    Write-Host "`n=== $($instance.Instance) ===" -ForegroundColor Cyan

    $failed = Get-DbaAgentJobHistory -SqlInstance $instance.Instance `
        -StartDate (Get-Date).AddHours(-$JobHistoryHours) `
        -ExcludeJobSteps |
        Where-Object {
            $_.Status -eq 'Failed' -and
            $_.Job -notin $ExcludeJobs
        } |
        Select-Object Job, RunDate, Duration, Message |
        Sort-Object RunDate -Descending

    if ($failed) {
        $failed |
            Format-Table -AutoSize |
            Out-String -Width 512 |
            Write-Host
    } else {
        Write-Host "  No failed jobs in the last $JobHistoryHours hours" -ForegroundColor Green
    }
}

### Currently Running Jobs Exceeding Expected Duration

Flags jobs currently executing longer than `$JobRunningMultiplier`x their average
historical duration. A job with no history is excluded since there is no baseline to
compare against.

In [ ]:
# Long Running Jobs
foreach ($instance in $SqlInstances) {
    Write-Host "`n=== $($instance.Instance) ===" -ForegroundColor Cyan

    $running = Get-DbaRunningJob -SqlInstance $instance.Instance |
        Where-Object { $_.Name -notin $ExcludeJobs }

    if (-not $running) {
        Write-Host "  No jobs currently running" -ForegroundColor Green
        continue
    }

    $flagged = foreach ($job in $running) {
        $history = Get-DbaAgentJobHistory -SqlInstance $instance.Instance `
            -Job $job.Name -ExcludeJobSteps |
            Where-Object { $_.Status -eq 'Succeeded' }

        if (-not $history) { continue }

        $avgSeconds  = ($history | Measure-Object { $_.Duration.TotalSeconds } -Average).Average
        $currentSeconds = (New-TimeSpan -Start $job.LastRunDate -End (Get-Date)).TotalSeconds
        $threshold   = $avgSeconds * $JobRunningMultiplier

        if ($currentSeconds -gt $threshold) {
            [PSCustomObject]@{
                JobName         = $job.Name
                StartDate       = $job.LastRunDate
                CurrentDuration = [timespan]::FromSeconds([math]::Round($currentSeconds, 0))
                AvgDuration     = [timespan]::FromSeconds([math]::Round($avgSeconds, 0))
                Multiplier      = [math]::Round($currentSeconds / $avgSeconds, 1)
            }
        }
    }

    if ($flagged) {
        $flagged |
            Format-Table -AutoSize |
            Out-String -Width 512 |
            Write-Host
    } else {
        Write-Host "  No jobs exceeding ${JobRunningMultiplier}x average duration" -ForegroundColor Green
    }
}